In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pykrx import stock
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# 1. 데이터 로드 및 전처리 (삼성전자)
ticker = "005930"
start_date = "20210101"
end_date = "20260501" # 오늘 날짜 기준

# 가격 및 수급 데이터 수집
df_price = stock.get_market_ohlcv_by_date(start_date, end_date, ticker)[['종가']]
df_investor = stock.get_market_net_purchases_of_equities_by_ticker(start_date, end_date, ticker)
try:
    df_investor = df_investor[['외국인', '기관합계']]
except KeyError:
    print("컬럼명이 다릅니다! 현재 컬럼명:", df_investor.columns)
df = pd.concat([df_price, df_investor], axis=1)
df.columns = ['Close', 'Foreign', 'Institution']

# 스케일링 (0~1 사이 정규화)
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df)

# 2. 데이터 셋 구성 (Windowing)
def create_sequences(data, seq_length):
    x, y = [], []
    for i in range(len(data) - seq_length):
        x.append(data[i:i+seq_length])
        y.append(data[i+seq_length, 0]) # 타겟은 다음날 종가
    return np.array(x), np.array(y)

seq_length = 60
X, y = create_sequences(df_scaled, seq_length)

# 텐서 변환 및 MPS/CPU 장치 설정
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
X_train = torch.FloatTensor(X).to(device)
y_train = torch.FloatTensor(y).view(-1, 1).to(device)

# 3. LSTM 모델 정의
class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(StockLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # [중요] 초기 상태 텐서도 입력 텐서와 동일한 장치(MPS/CPU)에 할당
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

# 모델 생성 (특징 3개: 종가, 외인, 기관)
model = StockLSTM(input_size=3, hidden_size=64, num_layers=2, output_size=1).to(device)

# 4. 학습 루프
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(100):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.6f}')

# 5. 결과 시각화
model.eval()
with torch.no_grad():
    pred = model(X_train).cpu().numpy()

# 역정규화 (가짜 배열 활용)
pred_final = scaler.inverse_transform(np.concatenate([pred, np.zeros((len(pred), 2))], axis=1))[:, 0]
actual_final = scaler.inverse_transform(np.concatenate([y.reshape(-1, 1), np.zeros((len(y), 2))], axis=1))[:, 0]

plt.figure(figsize=(10, 5))
plt.plot(actual_final, label='Actual')
plt.plot(pred_final, label='Predicted')
plt.title('Samsung Electronics Stock Prediction with Foreign/Inst.')
plt.legend()
plt.show()

Error occurred in get_market_net_purchases_of_equities_by_ticker: '005930'
컬럼명이 다릅니다! 현재 컬럼명: RangeIndex(start=0, stop=0, step=1)


ValueError: Length mismatch: Expected axis has 1 elements, new values have 3 elements

In [10]:
# 1. 데이터 로드 및 전처리 (삼성전자)
ticker = "005930"
start_date = "20210101"
end_date = "20260428" # 오늘 날짜 기준

# 가격 및 수급 데이터 수집
df_price = stock.get_market_ohlcv_by_date(start_date, end_date, ticker)[['종가']]
print(f"데이터 수집 중: {ticker}...")
try:
    # 가장 안정적인 함수로 시도
    df_investor = stock.get_market_net_purchases_of_equities_by_ticker(start_date, end_date, ticker)
    
    # 만약 빈 데이터가 왔다면 (RangeIndex 0 이슈 해결)
    if len(df_investor) == 0:
        print("첫 번째 함수 실패, 대체 함수로 재시도합니다.")
        # 대체 함수: 날짜별로 직접 긁어오기
        df_investor = stock.get_market_net_purchases(start_date, end_date, ticker)
except Exception as e:
    print(f"수급 데이터 수집 중 오류 발생: {e}")
    # 최후의 수단: 빈 값 대신 0으로 채운 더미 데이터 생성 (학습 중단을 막기 위함)
    df_investor = pd.DataFrame(0, index=df_price.index, columns=['외국인', '기관합계'])

데이터 수집 중: 005930...
Error occurred in get_market_net_purchases_of_equities_by_ticker: '005930'
첫 번째 함수 실패, 대체 함수로 재시도합니다.
수급 데이터 수집 중 오류 발생: module 'pykrx.stock' has no attribute 'get_market_net_purchases'
